# Niko Rada — Offline Safety Companion — Kaggle Demo

**Niko Rada** ("I'm safe" / "I'm calm" in Sheng/Kiswahili) is an offline-first Flutter app that runs Gemma 4 E2B
entirely on-device to give grounded, cited safety guidance during protests, network blackouts, and emergencies in
Kenya — with zero network calls at inference time.

This notebook is the **submission demo**. It is not a separate implementation: it loads the exact same
`assets/corpus/cards.jsonl` knowledge base the app ships with, and calls the exact same prompt templates from
`lib/prompts/prompts.dart` and the exact same retrieval algorithm from `lib/services/rag.dart`, ported to Python
line-for-line. Every markdown cell below that says "mirrors `lib/...`" is naming the literal source file this cell
reproduces — that equivalence is the point of this notebook.

**Why a notebook and not the on-device build for this demo:** the app's real engine (`flutter_gemma` +
`flutter_gemma_litertlm`, loading the `.litertlm` build of Gemma 4 E2B via LiteRT-LM FFI) is built for Android/desktop
native inference, not for a hosted Jupyter kernel. Kaggle doesn't run the Android/Linux native LiteRT-LM stack, so
this notebook loads the same Gemma 4 E2B model family through `transformers` instead — same model, same corpus,
same prompts, different loader. The golden rule (zero network calls at inference time) is an architectural property of the shipped app — no cloud SDK, no network in the request path. On-device inference is not yet verified on hardware: our only test phone is 4 GB (below the E2B floor), and the Linux desktop build currently fails inside the LiteRT-LM FFI native load. This notebook's job is therefore to demonstrate the app's reasoning end-to-end in a reviewable, runnable form.

**What this notebook demonstrates, in order:**
1. The real corpus (41 verified bilingual knowledge cards)
2. Retrieval (`lib/services/rag.dart` ported to Python)
3. Prompts (`lib/prompts/prompts.dart` ported to Python)
4. Grounded, cited answers for all 5 priority panic-button scenarios
5. The same query answered in English, Kiswahili, and Sheng
6. The refusal path — an out-of-corpus question gets "no verified guidance," not an improvised answer
7. Incident extraction — messy free text → strict JSON, matching the app's incident logger schema

Run all cells top-to-bottom. No manual steps, no file uploads — the corpus is embedded directly in this notebook.


## Setup

Install/upgrade `transformers` — Gemma 4 is a recent model family and needs a current version.

In [1]:
!pip install -q -U "transformers>=4.46" accelerate bitsandbytes
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


## 1. The corpus

This is a **byte-identical copy** of [`assets/corpus/cards.jsonl`](../assets/corpus/cards.jsonl) from the app repo
— 41 bilingual (English/Kiswahili) knowledge cards across 6 domains, each citing a real authoritative source
(ICRC, EFF Surveillance Self-Defense, Kenya Red Cross, the Constitution of Kenya 2010, KHRC, LSK, IMLU, Amnesty
Kenya, KNCHR, Missing Voices, Signal, Briar). Every contact number in the corpus was verified against the
organisation's own website (or, where automated access was blocked, against multiple independent reputable news
sources) — see `assets/corpus/SOURCES.md` for the full verification log, including two numbers that were found to
be wrong in an earlier pass and corrected.

Cards marked `"verified": false` would be excluded from retrieval automatically (matching `rag.dart`'s behaviour)
— every card currently in the corpus is verified, so none are excluded here, but the filter is applied for
correctness.

In [2]:
CARDS_JSONL = r"""
{"id":"teargas-001","domain":"first_aid","title_en":"Immediate response to tear gas","title_sw":"Hatua za haraka dhidi ya machozi ya kutoa","content_en":"1. Move UPWIND and UPHILL immediately — get to open air away from the cloud.\n2. Do NOT rub your eyes — rubbing spreads the agent and worsens burning.\n3. Blink rapidly to produce tears; flush with large amounts of clean water for 10–15 minutes if available.\n4. Remove contaminated clothing (top to bottom, avoiding the face); place in a sealed bag.\n5. Blow your nose and spit — do not swallow.\n6. If breathing is severely affected, sit upright and take slow deep breaths in fresh air.\n7. Seek medical help if breathing difficulty, severe eye pain, or skin blistering persists.","content_sw":"1. Songa KUELEKEA UPEPO na MAHALI PA JUU haraka — toka nje ya wingu la gesi.\n2. USISUGUE macho yako — kusugua kunaeneza kemikali na kuongeza maumivu.\n3. Blink haraka ili kutoa machozi; osha kwa maji mengi safi kwa dakika 10–15 ikiwa unapatikana.\n4. Vua nguo zilizochafuliwa (kutoka juu kwenda chini, epuka uso); weka kwenye mfuko uliofungwa.\n5. Piga mafua na temea mate — usimeze.\n6. Ikiwa kupumua ni ngumu sana, kaa wima na pumua polepole hewa safi.\n7. Tafuta msaada wa daktari ikiwa shida ya kupumua, maumivu makali ya macho, au malengelenge ya ngozi yanaendelea.","source":"ICRC / EFF Surveillance Self-Defense","source_url":"https://ssd.eff.org/module/attending-protest","region":"general","verified":true}
{"id":"teargas-002","domain":"first_aid","title_en":"Improvised tear gas protection","title_sw":"Ulinzi wa dharura dhidi ya gesi","content_en":"Before exposure: wear goggles (not glasses — they don't seal), N95 or folded wet cloth over nose/mouth, long sleeves.\nCarry: small bottle of water, saline solution, or diluted antacid (Maalox/milk of magnesia + water 50:50) — effective eye rinse.\nAvoid: contact lenses (trap agent against cornea). Avoid petroleum jelly on face (traps agent on skin).","content_sw":"Kabla ya kufunuliwa: vaa miwani ya kinga (si glasi za kawaida — hazifungi), N95 au kitambaa cha mvua kilichopigwa mbili kinywani/puani, mikono mirefu.\nBeba: chupa ndogo ya maji, suluhisho la saline, au antacid iliyopunguzwa (Maalox/maziwa ya magnesia + maji 50:50) — nzuri kwa kuosha macho.\nEpuka: lenzi za mawasiliano (zinazofunga kemikali kwenye jicho). Epuka petroleum jelly usoni (inabana kemikali ngozini).","source":"EFF Surveillance Self-Defense / Access Now","source_url":"https://ssd.eff.org/module/attending-protest","region":"general","verified":true}
{"id":"evasion-001","domain":"evasion_safety","title_en":"If you are being followed or targeted","title_sw":"Ikiwa unafuatwa au ukiwa shabaha","content_en":"1. Do NOT run — running signals panic and can escalate a threat.\n2. Move toward crowds, shops, churches, markets — populated, public, lit spaces.\n3. Change direction unpredictably; cross to the other side of the street.\n4. Enter a busy shop, explain quietly to a shopkeeper you are being followed.\n5. Call a trusted contact to your location if you have signal — share your GPS pin.\n6. Avoid entering vehicles with strangers or accepting 'help' from unfamiliar people.\n7. If you cannot reach safety, photograph or memorise details: vehicle plate, clothing, time, location.","content_sw":"1. USIKIMBIE — kukimbia kunaonyesha hofu na kunaweza kuongeza hatari.\n2. Elekea maeneo yenye watu, maduka, makanisa, masoko — maeneo ya wazi, ya umma, yenye mwanga.\n3. Badilisha mwelekeo bila mpangilio; vuka ng'ambo nyingine ya barabara.\n4. Ingia dukani lenye watu, eleza kwa utulivu kwa mwenye duka kwamba unafuatwa.\n5. Piga simu kwa mtu unayemwamini kwa mahali pako ikiwa una mtandao — tuma pin yako ya GPS.\n6. Epuka kuingia magari na wageni au kukubali 'msaada' kutoka kwa watu wasiojulikana.\n7. Ikiwa huwezi kufika mahali salama, piga picha au kumbuka maelezo: nambari ya gari, mavazi, wakati, mahali.","source":"Protection Briefing International / EFF SSD","source_url":"https://ssd.eff.org/module/attending-protest","region":"kenya","verified":true}
{"id":"evasion-002","domain":"evasion_safety","title_en":"Crowd crush and stampede safety","title_sw":"Usalama wakati wa msongamano na kukanyagana","content_en":"1. If caught in a crowd crush, cross your arms in front of your chest (like a boxer stance) — this protects your ribcage and keeps breathing space.\n2. Move diagonally to the edge of the crowd — not against the flow, diagonally.\n3. If you fall, curl into a ball protecting your head; shout loudly so others know you are on the ground.\n4. Avoid doorways, walls, barriers — the highest pressure points.\n5. If near tear gas AND a crush, prioritise getting out of the crush first — asphyxiation risk from crowd pressure is more immediate.","content_sw":"1. Ikiwa umefungwa katika msongamano, piga mikono yako mbele ya kifua (kama mkao wa mpigaboxing) — hii inalinda mbavu na kutoa nafasi ya kupumua.\n2. Hamia kwa pembe kwenye ukingo wa umati — si kinyume na mtiririko, bali kwa pembe.\n3. Ikiwa unaanguka, jikunje kama mpira ukikinga kichwa; piga kelele ili wengine wajue uko chini.\n4. Epuka milango, kuta, vizuizi — pointi zenye msongo mkubwa zaidi.\n5. Ikiwa uko karibu na gesi ya machozi NA msongamano, toka katika msongamano kwanza — hatari ya kukosa hewa kutokana na msongo ni ya haraka zaidi.","source":"ICRC / WHO Crowd Safety","source_url":"https://www.icrc.org","region":"general","verified":true}
{"id":"comms-001","domain":"blackout_comms","title_en":"Reaching people when the network is down","title_sw":"Kuwasiliana wakati mtandao umezimwa","content_en":"When mobile data and SMS are cut:\n1. WIFI CALLING: If you have Wi-Fi (café, building), WhatsApp, Signal, and Telegram can all call and message over Wi-Fi only — no mobile data needed.\n2. MESH APPS: Briar (Android, free) communicates via Bluetooth and Wi-Fi direct with nearby phones — works with NO internet. Install before the protest.\n3. OFFLINE SMS VIA SIGNAL: Signal can send SMS over regular cellular voice channel (not data) — enable in Signal settings as fallback.\n4. PRE-AGREED RENDEZVOUS: The most reliable plan needs no tech. Agree in advance: if separated, meet at [your agreed place] at [agreed time]. Write it down.\n5. FIND A LANDLINE: Hotels, hospitals, and offices often have landlines when mobile networks are overloaded or cut.","content_sw":"Wakati data ya simu na SMS zimezimwa:\n1. SIMU KWA WIFI: Ikiwa una Wi-Fi (mkahawa, jengo), WhatsApp, Signal, na Telegram zinaweza kupiga simu na kutuma ujumbe kupitia Wi-Fi peke yake — bila data ya simu.\n2. PROGRAMU ZA MESH: Briar (Android, bure) inawasiliana kupitia Bluetooth na Wi-Fi moja kwa moja na simu zilizo karibu — inafanya kazi BILA mtandao. Sakinisha kabla ya maandamano.\n3. SMS YA OFFLINE KWA SIGNAL: Signal inaweza kutuma SMS kupitia njia ya sauti ya kawaida ya simu (si data) — washa katika mipangilio ya Signal kama njia mbadala.\n4. MAHALI PA KUKUTANA KILICHOKUBALIWA: Mpango wa kuaminika zaidi hauhitaji teknolojia. Kubali mapema: ikiwa mtakapotengana, kukutaneni [mahali pako uliokubali] saa [wakati uliokubali]. Andika chini.\n5. TAFUTA SIMU YA KAWAIDA: Hoteli, hospitali, na ofisi mara nyingi zina simu za kawaida wakati mitandao ya simu imepakiwa au imezimwa.","source":"EFF SSD / Access Now Digital Security Helpline","source_url":"https://www.accessnow.org/help","region":"general","verified":true}
{"id":"comms-002","domain":"blackout_comms","title_en":"Rendezvous and family plan (prepare now)","title_sw":"Mpango wa kukutana na familia (jiandae sasa)","content_en":"Set this up BEFORE you go anywhere that might be disrupted:\n1. Choose TWO meeting points: one near your current location, one further (in case the first is unsafe).\n2. Agree on a CHECK-IN TIME: e.g. 'If I don't contact you by 6pm, meet me at [place].'\n3. Name an OUT-OF-AREA CONTACT: someone outside Nairobi who both parties can reach — they relay messages between you.\n4. Write all this on paper — not just in your phone. Phones die, get seized, or run out of battery.\n5. Share your plan with at least two trusted people before you leave.","content_sw":"Weka mpango huu KABLA ya kwenda popote ambapo kunaweza kuwa na msongo:\n1. Chagua MAENEO MAWILI ya kukutana: moja karibu na mahali ulipo sasa, lingine mbali zaidi (ikiwa la kwanza si salama).\n2. Kubali WAKATI WA KURIPOTI: mfano 'Ikiwa sijakufikia saa 12 usiku, nikusandie [mahali].'\n3. Taja MAWASILIANO YA NJE YA MAENEO: mtu nje ya Nairobi ambaye pande zote mbili zinaweza kumfikia — wao watawasiliana ujumbe kati yenu.\n4. Andika haya yote kwa karatasi — si tu kwenye simu yako. Simu zinakufa, zinakamatwa, au zinakwisha betri.\n5. Shiriki mpango wako na watu wawili wanaokuamini kabla hujaenda.","source":"Access Now / EFF SSD","source_url":"https://ssd.eff.org/module/attending-protest","region":"general","verified":true}
{"id":"doc-001","domain":"documentation","title_en":"Witnessing an arrest or abduction — what to record","title_sw":"Kushuhudia kukamatwa au kutekwa — nini cha kurekodi","content_en":"If you witness someone being arrested or forced into a vehicle:\nRECORD (if safe to do so):\n- Time and exact location (street name, nearest landmark, GPS if possible)\n- Description of the person taken: approximate age, gender, clothing, any identifying features\n- Number and description of vehicles: colour, make, plate number (even partial)\n- Number and description of officers/perpetrators: uniform or plain-clothes, any badges or insignia\n- What was said, if anything\n- Names of any witnesses\nDO NOT:\n- Put yourself in danger to record\n- Share publicly before consulting a lawyer or human rights organisation\nSTORE SECURELY: Back up to a trusted person or encrypted cloud when you have signal.","content_sw":"Ikiwa unashuhudia mtu akikamatwa au kulazimishwa kwenye gari:\nRECORDI (ikiwa ni salama kufanya hivyo):\n- Wakati na mahali halisi (jina la barabara, alama ya karibu, GPS ikiwezekana)\n- Maelezo ya mtu aliyechukuliwa: umri wa karibu, jinsia, mavazi, sifa yoyote ya kutambulisha\n- Idadi na maelezo ya magari: rangi, aina, nambari ya bati (hata sehemu)\n- Idadi na maelezo ya maafisa/wahusika: sare au mavazi ya kawaida, beji au alama yoyote\n- Kilichosemwa, kama kilichosemwa\n- Majina ya mashahidi wowote\nUSIfanye:\n- Jihatarisha kupiga picha\n- Shiriki hadharani kabla ya kushauriana na wakili au shirika la haki za binadamu\nHIFADHI KWA USALAMA: Rudisha kwa mtu unayemwamini au wingu lililofichwa unapopata mtandao.","source":"Missing Voices Kenya / KHRC / Amnesty International","source_url":"https://www.missingvoices.net","region":"kenya","verified":true}
{"id":"doc-002","domain":"documentation","title_en":"Reporting to human rights organisations in Kenya","title_sw":"Kuripoti kwa mashirika ya haki za binadamu Kenya","content_en":"When you have signal, you can report to:\n- MISSING VOICES: missingvoices.or.ke — database of disappearances and unlawful killings in Kenya; report via their 'Report Incident' page\n- KHRC (Kenya Human Rights Commission): khrc.or.ke | +254 20 204 4545 or +254 20 210 6763 (verify on official site)\n- AMNESTY INTERNATIONAL KENYA: amnestykenya.org | +254 759 464 346 (verify on official site)\n- LSK (Law Society of Kenya): lsk.or.ke | +254 799 595 800 — for legal referrals and detainee tracking\n- KNCHR (Kenya National Commission on Human Rights): knchr.org | 0800 720 627 (toll-free) or SMS 22359\n- IMLU (Independent Medico-Legal Unit): imlu.org | 0800 721 401 (toll-free) — documents torture and ill-treatment\nNOTE: Verify all contact numbers on official websites before calling. Numbers change.","content_sw":"Unapopata mtandao, unaweza kuripoti kwa:\n- MISSING VOICES: missingvoices.or.ke — hifadhidata ya kutoweka na mauaji yasiyo halali Kenya; ripoti kupitia ukurasa wao wa 'Report Incident'\n- KHRC (Tume ya Haki za Binadamu Kenya): khrc.or.ke | +254 20 204 4545 au +254 20 210 6763 (thibitisha kwenye tovuti rasmi)\n- AMNESTY INTERNATIONAL KENYA: amnestykenya.org | +254 759 464 346 (thibitisha kwenye tovuti rasmi)\n- LSK (Jumuiya ya Sheria Kenya): lsk.or.ke | +254 799 595 800 — kwa rufaa za kisheria na ufuatiliaji wa wafungwa\n- KNCHR: knchr.org | 0800 720 627 (bila malipo) au SMS 22359\n- IMLU: imlu.org | 0800 721 401 (bila malipo) — inadokumenta mateso na ukatili\nNOTA: Thibitisha nambari zote za mawasiliano kwenye tovuti rasmi kabla ya kupiga simu. Nambari zinabadilika.","source":"KHRC / Amnesty Kenya / LSK / KNCHR / IMLU / Missing Voices","source_url":"https://www.khrc.or.ke","region":"kenya","verified":true}
{"id":"phone-001","domain":"documentation","title_en":"Prepare your phone before a protest","title_sw":"Andaa simu yako kabla ya maandamano","content_en":"BEFORE you go:\n1. CHARGE FULLY and carry a power bank.\n2. NOTE DOWN critical numbers on paper — if your phone is seized, you still have them.\n3. ENABLE ENCRYPTION: Android Settings > Security > Encryption (most modern phones are encrypted by default).\n4. SET A STRONG PIN (not biometric at the lock screen) — you can be legally compelled to use your fingerprint, not a PIN in many jurisdictions.\n5. DISABLE LOCK-SCREEN NOTIFICATIONS — so messages are not visible without unlocking.\n6. BACK UP photos, contacts, and messages to a trusted encrypted location.\n7. REVIEW and remove sensitive content: old protest photos, group chats, contacts of organisers if the risk is high.\n8. INSTALL Briar (mesh messaging) and Signal while you have Wi-Fi.\n9. NOTE your IMEI (dial *#06#) — proof of ownership if seized.","content_sw":"KABLA hujaenda:\n1. CHAJI KIKAMILIFU na beba power bank.\n2. ANDIKA nambari muhimu kwa karatasi — ikiwa simu yako itakamatwa, bado unanazo.\n3. WESHA USIMBUAJI: Mipangilio ya Android > Usalama > Usimbuaji (simu nyingi za kisasa zimesimbwa kwa chaguo-msingi).\n4. WEKA NAMBARI NGUMU YA PIN (si alama ya kidole kwenye skrini ya kufunga) — unaweza kulazimishwa kisheria kutumia kidole chako, si PIN katika maeneo mengi.\n5. ZIMA arifa za skrini ya kufunga — ili ujumbe usionekane bila kufungua.\n6. HIFADHI picha, anwani, na ujumbe mahali salama pa siri.\n7. KAGUA na uondoe maudhui nyeti: picha za zamani za maandamano, mazungumzo ya kikundi, anwani za wapangaji ikiwa hatari ni kubwa.\n8. SAKINISHA Briar (ujumbe wa mesh) na Signal ukiwa na Wi-Fi.\n9. KUMBUKA IMEI yako (piga *#06#) — uthibitisho wa umiliki ikiwa itakamatwa.","source":"EFF Surveillance Self-Defense / Access Now","source_url":"https://ssd.eff.org/module/your-security-plan","region":"general","verified":true}
{"id":"rights-001","domain":"rights","title_en":"Your rights if arrested in Kenya","title_sw":"Haki zako ukikamatwa Kenya","content_en":"Under the Constitution of Kenya 2010, Article 49:\n1. You have the right to be INFORMED PROMPTLY of the reason for your arrest.\n2. You have the right to REMAIN SILENT and not to incriminate yourself.\n3. You have the right to COMMUNICATE with an advocate (lawyer) or a person you choose.\n4. You must be brought before a court within 24 HOURS of arrest, or the next working day if 24 hours falls on a holiday.\n5. You have the right to be treated with DIGNITY and not to be subjected to cruel, inhumane, or degrading treatment.\n6. Demand to know the NAME and BADGE NUMBER of any officer dealing with you.\n7. DOCUMENT everything you can remember as soon as possible after release.\nIf your rights are violated, report to LSK, KHRC, or KNCHR.","content_sw":"Chini ya Katiba ya Kenya 2010, Kifungu 49:\n1. Una haki ya KUJULISHWA HARAKA sababu ya kukamatwa kwako.\n2. Una haki ya KUKAA KIMYA na kutojishitaki.\n3. Una haki ya KUWASILIANA na wakili au mtu unayemchagua.\n4. Lazima uletwe mahakamani ndani ya MASAA 24 ya kukamatwa, au siku inayofuata ya kazi ikiwa masaa 24 yanaangukia likizo.\n5. Una haki ya kutendwa kwa HESHIMA na kutofanyiwa ukatili, udhalilishaji, au matibabu ya aibu.\n6. Dai kujua JINA na NAMBARI YA BEJI ya afisa yeyote anayekushughulika nawe.\n7. DOKUMETA kila kitu unachoweza kukumbuka haraka iwezekanavyo baada ya kuachiwa.\nIkiwa haki zako zimekiukwa, ripoti kwa LSK, KHRC, au KNCHR.","source":"Constitution of Kenya 2010, Article 49 / LSK","source_url":"https://www.lsk.or.ke","region":"kenya","verified":true}
{"id":"rights-002","domain":"rights","title_en":"Your right to protest in Kenya","title_sw":"Haki yako ya kuandamana Kenya","content_en":"The Constitution of Kenya 2010 protects:\n- Article 33: Freedom of expression\n- Article 36: Freedom of association\n- Article 37: Right to assemble, demonstrate, picket, and petition\nThese rights are NOT absolute — they may be limited by law to the extent the limitation is reasonable and justifiable. However:\n- You do NOT need a permit to hold a peaceful protest (High Court, 2016).\n- Police may not disperse a peaceful assembly without lawful reason.\n- The use of live ammunition against protesters is illegal except in extreme and documented circumstances.\n- A May 2025 High Court order bars internet shutdowns during protests.\nIf you believe your rights have been violated, contact LSK or KHRC.","content_sw":"Katiba ya Kenya 2010 inalinda:\n- Kifungu 33: Uhuru wa kujieleza\n- Kifungu 36: Uhuru wa kujumuika\n- Kifungu 37: Haki ya kukusanyika, kuandamana, kupiga mstari, na kuomba\nHaki hizi SI KAMILI — zinaweza kupunguzwa na sheria kwa kiwango ambacho upunguzaji ni wa busara na unaoweza kuhesabiwa. Hata hivyo:\n- HUHITAJI kibali cha kushikilia maandamano ya amani (Mahakama Kuu, 2016).\n- Polisi haiwezi kuvunja mkutano wa amani bila sababu ya kisheria.\n- Matumizi ya risasi za moto dhidi ya waandamanaji ni kinyume cha sheria isipokuwa katika mazingira ya kipekee na yaliyodokumentwa.\n- Amri ya Mahakama Kuu ya Mei 2025 inapiga marufuku kuzima mtandao wakati wa maandamano.\nIkiwa unaamini haki zako zimekiukwa, wasiliana na LSK au KHRC.","source":"Constitution of Kenya 2010 / LSK / High Court Ruling 2016","source_url":"https://www.lsk.or.ke","region":"kenya","verified":true}
{"id":"firstaid-001","domain":"first_aid","title_en":"Rubber bullet and baton injuries — first response","title_sw":"Majeraha ya risasi za mpira na fimbo — jibu la kwanza","content_en":"Rubber bullets / baton strikes:\n1. STOP THE BLEEDING: apply firm direct pressure with a clean cloth. Maintain for at least 10 minutes without lifting.\n2. Do NOT remove an embedded object — stabilise it in place.\n3. Suspected FRACTURE: immobilise the limb. Do not attempt to set it. Support with improvised splint (stiff material + cloth).\n4. HEAD INJURY: treat as serious. Lay the person down, keep still, do not give food or drink, monitor consciousness. Get medical help.\n5. TORSO injury from rubber bullet at close range: treat as serious internal injury — get medical help immediately.\n6. Kenya Red Cross Ambulance: 1199 (toll-free, 24/7) — verify on official site.","content_sw":"Risasi za mpira / mapigo ya fimbo:\n1. SIMAMISHA DAMU: weka shinikizo imara la moja kwa moja kwa kitambaa safi. Shikilia kwa angalau dakika 10 bila kuinua.\n2. USIONDOE kitu kilichopachikwa — kiimarisha mahali pake.\n3. FRACTURE inayoshukiwa: imarisha kiungo. Usijaribu kuiweka. Saidia kwa banzi la kuimprovisa (nyenzo ngumu + kitambaa).\n4. JERAHA LA KICHWA: lichukulie kwa uzito. Mlaze mtu chini, mshike tulivu, usimpe chakula wala maji, angalia ufahamu. Tafuta msaada wa daktari.\n5. Jeraha la KIFUA kutoka kwa risasi ya mpira kwa umbali mfupi: lichukulie kama jeraha kubwa la ndani — tafuta msaada wa daktari mara moja.\n6. Kenya Red Cross Ambulance: 1199 (bila malipo, saa 24) — thibitisha kwenye tovuti rasmi.","source":"ICRC First Aid / Kenya Red Cross","source_url":"https://redcross.or.ke","region":"kenya","verified":true}
{"id":"firstaid-002","domain":"first_aid","title_en":"Shock — recognising and responding","title_sw":"Mshtuko — kutambua na kujibu","content_en":"Signs of shock (dangerous — act quickly):\n- Pale, clammy, cold skin\n- Rapid weak pulse\n- Fast shallow breathing\n- Confusion, anxiety, or unresponsiveness\n- Dizziness or fainting\nResponse:\n1. Lay the person down; raise legs about 30cm (unless head, neck, or chest injury — then don't).\n2. Keep them warm — use a jacket or blanket.\n3. Do NOT give food, water, or alcohol.\n4. Talk calmly to keep them conscious.\n5. Do NOT leave them alone.\n6. Get emergency medical help as fast as possible.\nShock can be life-threatening. This is a medical emergency.","content_sw":"Dalili za mshtuko (hatari — tenda haraka):\n- Ngozi nyeupe, baridi, yenye jasho\n- Mapigo ya moyo ya haraka na dhaifu\n- Kupumua haraka na kwa kina kidogo\n- Mkanganyiko, wasiwasi, au kutokujibu\n- Kizunguzungu au kuzirai\nJibu:\n1. Mlaze mtu chini; inua miguu kwa sentimita 30 (isipokuwa kuna jeraha la kichwa, shingo, au kifua — basi usifanye).\n2. Mweke joto — tumia koti au blanketi.\n3. USIMPE chakula, maji, au pombe.\n4. Zungumza kwa utulivu ili amweke macho.\n5. USIMWAACHE peke yake.\n6. Tafuta msaada wa daktari wa dharura haraka iwezekanavyo.\nMshtuko unaweza kuhatarisha maisha. Hii ni dharura ya kimatibabu.","source":"ICRC First Aid / Kenya Red Cross","source_url":"https://www.redcross.or.ke","region":"general","verified":true}
{"id":"evasion-003","domain":"evasion_safety","title_en":"If you think your phone is being monitored","title_sw":"Ikiwa unafikiria simu yako inafuatiliwa","content_en":"Signs of potential surveillance: unusual battery drain, phone warm when idle, data usage spikes, unfamiliar apps.\nImmediate steps:\n1. Do not discuss sensitive plans on any call or message app on that device.\n2. Switch to Signal for communications — it provides end-to-end encryption and disappearing messages.\n3. Enable AIRPLANE MODE when not actively needed — stops most real-time tracking.\n4. Avoid using mobile data when stationary — prefer Wi-Fi in trusted locations.\n5. Do not click links in unsolicited messages — phishing is a primary infection vector.\n6. For very sensitive situations: use a secondary inexpensive phone with a new SIM bought with cash, kept powered off until needed.\nSource: EFF SSD — for more detail, see ssd.eff.org","content_sw":"Ishara za ufuatiliaji unaowezekana: betri inayoisha haraka kwa kawaida, simu joto wakati haifanyi kazi, ongezeko la data, programu zisizojulikana.\nHatua za haraka:\n1. Usizungumzie mipango nyeti kwa simu au programu yoyote ya ujumbe kwenye kifaa hicho.\n2. Badilisha kwa Signal kwa mawasiliano — hutoa usimbuaji wa mwisho-hadi-mwisho na ujumbe unaopotea.\n3. Wesha HALI YA NDEGE unapohitajika — inasimamisha ufuatiliaji wa wakati halisi.\n4. Epuka kutumia data ya simu ukiwa kimya — pendelea Wi-Fi katika maeneo unayoyaamini.\n5. Usibofye viungo katika ujumbe usiotarajiwa — uvuvi wa wadanganyifu ni njia kuu ya maambukizi.\n6. Kwa hali nyeti sana: tumia simu ya pili ya bei nafuu na SIM mpya iliyonunuliwa kwa pesa taslimu, iliyowekwa imezimwa hadi inapohitajika.","source":"EFF Surveillance Self-Defense","source_url":"https://ssd.eff.org","region":"general","verified":true}
{"id":"contacts-001","domain":"contacts","title_en":"Emergency contacts — Kenya (verified)","title_sw":"Nambari za dharura — Kenya (zilizothibitishwa)","content_en":"EMERGENCY (police, fire, ambulance): 999 or 112\nKENYA RED CROSS AMBULANCE: 1199 (toll-free, 24/7 nationwide)\nNAIROBI HOSPITAL EMERGENCY: +254 20 284 5000\nAGA KHAN UNIVERSITY HOSPITAL (Nairobi) Adult Casualty: +254 20 366 2025 | Main: +254 20 366 2000\nIMPORTANT: Mobile networks may be down. Landlines at hospitals/hotels often still work. Know these by heart or on paper.\nNOTE: Always verify contacts at official websites. Numbers may change.","content_sw":"DHARURA (polisi, moto, ambulance): 999 au 112\nKENYA RED CROSS AMBULANCE: 1199 (bila malipo, saa 24 nchi nzima)\nHOSPITALI YA NAIROBI DHARURA: +254 20 284 5000\nHOSPITALI YA AGA KHAN UNIVERSITY (Nairobi) Casualty ya Watu Wazima: +254 20 366 2025 | Kuu: +254 20 366 2000\nMUHIMU: Mitandao ya simu inaweza kuwa imezimwa. Simu za kawaida za hospitali/hoteli mara nyingi bado zinafanya kazi. Jua hizi kwa moyo au kwa karatasi.\nNOTA: Thibitisha anwani kila wakati kwenye tovuti rasmi. Nambari zinaweza kubadilika.","source":"Kenya Red Cross / Nairobi Hospital / Aga Khan Hospital","source_url":"https://redcross.or.ke","region":"kenya","verified":true}
{"id":"rights-003","domain":"rights","title_en":"If someone you know has been arrested or disappeared","title_sw":"Ikiwa mtu unayemjua amekamatwa au kutoweka","content_en":"Immediate steps:\n1. Note the time, location, and circumstances you last had contact.\n2. Contact LSK (lsk.or.ke) for legal referral — they can help locate detained persons.\n3. File a report with Missing Voices (missingvoices.net) — this creates a documented record.\n4. Contact KHRC or Amnesty Kenya — they track patterns and can apply pressure.\n5. If you believe there is imminent risk to life, contact Kenya Red Cross or go directly to a police station to file a missing persons report (get the OB number — the Occurrence Book number — as a reference).\n6. Preserve all communications — do not delete messages, call logs, or photos.\n7. Do not pay for release without legal advice — this can constitute a crime.","content_sw":"Hatua za haraka:\n1. Kumbuka wakati, mahali, na mazingira ya mwisho kuwasiliana nawe.\n2. Wasiliana na LSK (lsk.or.ke) kwa rufaa ya kisheria — wanaweza kusaidia kupata watu waliokamatwa.\n3. Weka ripoti na Missing Voices (missingvoices.net) — hii inaunda rekodi iliyodokumentwa.\n4. Wasiliana na KHRC au Amnesty Kenya — wanafuatilia mifumo na wanaweza kutumia shinikizo.\n5. Ikiwa unaamini kuna hatari ya haraka kwa maisha, wasiliana na Kenya Red Cross au nenda moja kwa moja katika kituo cha polisi kuweka ripoti ya mtu aliyepotea (pata nambari ya OB — Occurrence Book — kama kumbukumbu).\n6. Hifadhi mawasiliano yote — usiifute ujumbe, kumbukumbu za simu, au picha.\n7. Usilipia kuachiwa bila ushauri wa kisheria — hii inaweza kuwa uhalifu.","source":"LSK / Missing Voices / KHRC / Amnesty Kenya","source_url":"https://www.missingvoices.net","region":"kenya","verified":true}
{"id":"comms-003","domain":"blackout_comms","title_en":"Using Briar for offline messaging","title_sw":"Kutumia Briar kwa ujumbe wa nje ya mtandao","content_en":"Briar is a free Android app that sends messages via Bluetooth and Wi-Fi Direct — NO internet needed.\nHow to use:\n1. Both people must have Briar installed (install NOW while you have Wi-Fi).\n2. Add each other as contacts in Briar while online — you only need internet to add contacts, not to message.\n3. When networks are down, open Briar — it will automatically find and connect to nearby Briar users via Bluetooth/Wi-Fi Direct.\n4. Range: roughly 10–100 metres for direct connection; extends through a mesh if multiple people have Briar.\n5. Briar also works over Tor for extra anonymity when internet is available.\nDownload: search 'Briar' on Google Play or F-Droid. It is free and open source.","content_sw":"Briar ni programu ya Android ya bure ambayo inatuma ujumbe kupitia Bluetooth na Wi-Fi Direct — BILA mtandao.\nJinsi ya kutumia:\n1. Watu wote wawili lazima wawe na Briar imewekwa (iweke SASA ukiwa na Wi-Fi).\n2. Ongezeni marafiki wako kwa Briar ukiwa mtandaoni — unahitaji mtandao tu kuongeza marafiki, si kutuma ujumbe.\n3. Mitandao ikizimwa, fungua Briar — itaungana kiotomatiki na watumiaji wa Briar wa karibu kupitia Bluetooth/Wi-Fi Direct.\n4. Umbali: takriban mita 10–100 kwa unganiko la moja kwa moja; unaweza kupanuka kupitia mesh ikiwa watu wengi wana Briar.\n5. Briar pia inafanya kazi kupitia Tor kwa kutokujulikana zaidi wakati mtandao unapatikana.\nPakua: tafuta 'Briar' kwenye Google Play au F-Droid. Ni bure na chanzo wazi.","source":"Briar Project / EFF SSD","source_url":"https://briarproject.org","region":"general","verified":true}
{"id":"evasion-004","domain":"evasion_safety","title_en":"If you are stopped at a checkpoint","title_sw":"Ikiwa unasimamishwa katika kizuizi","content_en":"Stay calm. De-escalation is your goal.\n1. Stop and comply with the instruction to halt.\n2. Keep your hands visible. Move slowly and explain what you are doing before doing it ('I am going to reach for my ID').\n3. You have the right to ask: 'Am I free to go?' — do this politely.\n4. You do NOT have to unlock your phone or give your PIN. Under Article 31 of the Constitution, you have a right to privacy of your personal data.\n5. Do NOT argue, resist physically, or make sudden movements.\n6. If detained: ask to call a lawyer or a trusted contact immediately (Article 49 right).\n7. Memorise or note the officer's name/badge number if visible.\n8. After the encounter: document everything you can remember as soon as you are safe.","content_sw":"Kaa tulivu. Kupunguza msongo ndilo lengo lako.\n1. Simama na fuata amri ya kusimama.\n2. Weka mikono yako ionekane. Sogea polepole na eleza unachofanya kabla ya kufanya ('Ninaenda kufikia kitambulisho changu').\n3. Una haki ya kuuliza: 'Niko huru kwenda?' — fanya hivi kwa adabu.\n4. HUHITAJI kufungua simu yako au kutoa PIN yako. Chini ya Kifungu 31 cha Katiba, una haki ya faragha ya data yako ya kibinafsi.\n5. USISHINDANE, upinzane kimwili, au ufanye harakati za ghafla.\n6. Ukizuiwa: omba kupiga simu kwa wakili au mawasiliano unayoyaamini mara moja (haki ya Kifungu 49).\n7. Kumbuka au andika jina/nambari ya beji ya afisa ikiwa inaonekana.\n8. Baada ya kukutana: dokumeta kila kitu unachoweza kukumbuka mara tu unapokuwa salama.","source":"LSK / Constitution of Kenya 2010, Articles 31 & 49","source_url":"https://www.lsk.or.ke","region":"kenya","verified":true}
{"id":"firstaid-003","domain":"first_aid","title_en":"Severe bleeding — tourniquet and wound packing","title_sw":"Kutokwa na damu sana — kitambaa cha kuzuia damu na kujaza jeraha","content_en":"For life-threatening limb bleeding (blood spurting, soaking through bandages):\nTOURNIQUET (limb only, never torso or neck):\n1. Apply 5–7 cm above the wound (not on a joint).\n2. Tighten until bleeding stops — it will be painful. That's correct.\n3. NOTE THE TIME it was applied — write it on the person's skin if possible.\n4. Do NOT remove a tourniquet in the field once applied.\nWOUND PACKING (torso/groin wounds where tourniquet is impossible):\n1. Pack the wound firmly with clean cloth, pushing deep into the cavity.\n2. Apply firm, continuous pressure.\n3. Do not remove the packing.\nGet the person to medical care as fast as possible.","content_sw":"Kwa kutokwa na damu kwa kiwango cha kuhatarisha maisha kwa kiungo (damu inayotoka kwa nguvu, kuloweka bandeji):\nKITAMBAA CHA KUZUIA DAMU (kwa kiungo tu, kamwe torso au shingo):\n1. Weka sentimita 5–7 juu ya jeraha (si kwenye pamoja).\n2. Kaza hadi damu ikome — itaumiza. Hiyo ni sawa.\n3. KUMBUKA WAKATI kiliwekwa — andika kwenye ngozi ya mtu ikiwa inawezekana.\n4. USIONDOE kitambaa cha kuzuia damu porini mara kilivyowekwa.\nKUJAZA JERAHA (majeraha ya torso/kinena ambapo kitambaa cha kuzuia damu haiwezekani):\n1. Jaza jeraha kwa nguvu na kitambaa safi, kikusukuma ndani kwenye cavity.\n2. Weka shinikizo imara na la kuendelea.\n3. Usiondoe kujaza.\nPeleka mtu kwa huduma ya daktari haraka iwezekanavyo.","source":"ICRC First Aid / Kenya Red Cross","source_url":"https://www.redcross.or.ke","region":"general","verified":true}
{"id":"comms-004","domain":"blackout_comms","title_en":"Signal app — setup for offline resilience","title_sw":"Programu ya Signal — usanidi kwa nguvu ya nje ya mtandao","content_en":"Signal is the most secure messaging app for protests.\nKey settings to enable NOW:\n1. DISAPPEARING MESSAGES: Set a timer (e.g. 1 week) so old messages auto-delete.\n2. NOTE TO SELF: Use as a secure notepad — notes are encrypted.\n3. REGISTRATION LOCK: Settings > Account > Registration Lock — prevents someone from re-registering your number.\n4. SCREEN LOCK: Settings > Privacy > Screen Lock.\n5. INCOGNITO KEYBOARD: Settings > Privacy — stops the keyboard from learning your words.\n6. SMS FALLBACK: Signal can send as SMS if data is unavailable (basic, unencrypted — but reaches people).\nDuring a blackout: Signal over Wi-Fi still works. Signal SMS still works on cellular voice channel.","content_sw":"Signal ndiyo programu ya ujumbe yenye usalama zaidi kwa maandamano.\nMipangilio muhimu ya kuwasha SASA:\n1. UJUMBE UNAOPOTEA: Weka kipima wakati (mfano wiki 1) ili ujumbe wa zamani ufutike kiotomatiki.\n2. KIDOKEZO KWA MIMI MWENYEWE: Tumia kama daftari salama — maelezo yamefichwa.\n3. KUFULI CHA USAJILI: Mipangilio > Akaunti > Kufuli cha Usajili — inazuia mtu yeyote kusajili upya nambari yako.\n4. KUFULI CHA SKRINI: Mipangilio > Faragha > Kufuli cha Skrini.\n5. KIBODI YA KUFICHA: Mipangilio > Faragha — inazuia kibodi kujifunza maneno yako.\n6. UHIFADHI WA SMS: Signal inaweza kutuma kama SMS ikiwa data haipatikani (ya msingi, bila usimbuaji — lakini inafikia watu).\nWakati wa kuzimwa: Signal kupitia Wi-Fi bado inafanya kazi. SMS ya Signal bado inafanya kazi kwenye njia ya sauti ya simu.","source":"Signal Foundation / EFF SSD","source_url":"https://signal.org","region":"general","verified":true}
{"id":"firstaid-004","domain":"first_aid","title_en":"Burns from flares or fires","title_sw":"Kuungua kutoka kwa moto au flare","content_en":"MINOR BURNS (redness, no blistering):\n1. Cool immediately with running cool (not cold/icy) water for 10–20 minutes.\n2. Do NOT apply toothpaste, butter, or oil — they trap heat and cause infection.\n3. Cover loosely with a clean dressing or cling film.\nSERIOUS BURNS (blistering, white/charred skin, larger than palm of hand, face/hands/genitals):\n1. Cool with water for 20 minutes.\n2. Do NOT burst blisters.\n3. Cover loosely — do NOT use fluffy materials (cotton wool) that stick.\n4. Treat for shock (person may be in shock even with small burns).\n5. Get emergency medical care immediately.\nDo NOT remove clothing stuck to a burn.","content_sw":"KUUNGUA KIDOGO (uwekundu, bila malengelenge):\n1. Poza mara moja na maji ya bomba ya baridi (si barafu) kwa dakika 10–20.\n2. USIWEKE dawa ya meno, siagi, au mafuta — yanabana joto na kusababisha maambukizi.\n3. Funika kwa uangalifu na bandeji safi au foil ya plastiki.\nKUUNGUA KWA KIASI KIKUBWA (malengelenge, ngozi nyeupe/iliyochomeka, kubwa kuliko mkono wa mtu, uso/mikono/sehemu za siri):\n1. Poza na maji kwa dakika 20.\n2. USIPASUE malengelenge.\n3. Funika kwa uangalifu — USITUMIE nyenzo za fluff (pamba) zinazoshikamana.\n4. Tibu kwa mshtuko (mtu anaweza kuwa katika mshtuko hata na kuungua kidogo).\n5. Tafuta huduma ya daktari wa dharura mara moja.\nUSIVUE nguo zilizoshikamana na kuungua.","source":"ICRC First Aid / Kenya Red Cross","source_url":"https://www.redcross.or.ke","region":"general","verified":true}
{"id": "choking-001", "domain": "first_aid", "title_en": "Choking — clearing an obstructed airway", "title_sw": "Kubanwa na kitu kooni — kufungua njia ya hewa", "content_en": "If someone is choking and CANNOT cough, speak, or breathe:\n1. Give up to 5 firm back blows between the shoulder blades with the heel of your hand, leaning them forward.\n2. If that doesn't clear it, give up to 5 abdominal thrusts (Heimlich): stand behind them, fist above the navel, grasp with your other hand, pull sharply inward and upward.\n3. Alternate 5 back blows and 5 abdominal thrusts until the object clears or they become unconscious.\n4. If they become unconscious, lower them to the ground carefully and begin CPR if trained; send someone to call for help immediately.\n5. Even after the object clears, they should be seen by a doctor — abdominal thrusts can cause internal injury.", "content_sw": "Ikiwa mtu amebanwa na kitu kooni na HAWEZI kukohoa, kuzungumza, au kupumua:\n1. Mpige mgongoni mara 5 kati ya mabega kwa kitako cha mkono wako, akiwa ameinama mbele.\n2. Ikiwa haisaidii, fanya msukumo wa tumbo (Heimlich) mara 5: simama nyuma yake, ngumi juu ya kitovu, shika kwa mkono mwingine, vuta kwa nguvu ndani na juu.\n3. Badilisha kati ya mapigo 5 ya mgongoni na misukumo 5 ya tumbo hadi kitu kitoke au apoteze fahamu.\n4. Akipoteza fahamu, mlaze chini kwa uangalifu na uanze CPR ikiwa umefunzwa; mtume mtu apige simu ya msaada mara moja.\n5. Hata kitu kikitoka, lazima aonwe na daktari — misukumo ya tumbo inaweza kusababisha jeraha la ndani.", "source": "ICRC First Aid", "source_url": "https://www.icrc.org/en/first-aid", "region": "general", "verified": true}
{"id": "heat-001", "domain": "first_aid", "title_en": "Heat exhaustion and dehydration during long protests", "title_sw": "Uchovu wa joto na upungufu wa maji wakati wa maandamano marefu", "content_en": "Signs: heavy sweating, weakness, dizziness, nausea, headache, cool clammy skin.\n1. Move to shade or a cooler area immediately.\n2. Loosen tight clothing.\n3. Sip water slowly — do not gulp large amounts at once.\n4. Cool the skin with a wet cloth on the neck, wrists, and armpits.\n5. If vomiting, confusion, very high body temperature, or no improvement in 30 minutes: this may be heat STROKE — a medical emergency. Get help immediately.\nPrevention: carry water, wear light-coloured loose clothing, take shade breaks.", "content_sw": "Dalili: jasho jingi, udhaifu, kizunguzungu, kichefuchefu, maumivu ya kichwa, ngozi baridi yenye jasho.\n1. Sogea kivulini au mahali penye baridi zaidi mara moja.\n2. Legeza nguo zilizobana.\n3. Kunywa maji polepole — usimeze kwa wingi mara moja.\n4. Poza ngozi kwa kitambaa chenye unyevu shingoni, kwenye vifundo vya mikono, na kwapani.\n5. Ikiwa kuna kutapika, mkanganyiko, joto kali la mwili, au hakuna nafuu ndani ya dakika 30: hii inaweza kuwa HEAT STROKE — dharura ya kimatibabu. Tafuta msaada mara moja.\nKinga: beba maji, vaa nguo nyepesi zenye rangi ya wazi, pumzika kivulini mara kwa mara.", "source": "ICRC / WHO Heat Stress Guidance", "source_url": "https://www.icrc.org/en/first-aid", "region": "general", "verified": true}
{"id": "unconscious-001", "domain": "first_aid", "title_en": "Unconscious but breathing — the recovery position", "title_sw": "Amepoteza fahamu lakini anapumua — mkao wa kupona", "content_en": "If someone is unconscious but breathing normally:\n1. Kneel beside them. Place the arm nearest you at a right angle to their body, palm up.\n2. Bring their far arm across their chest, back of hand against their near cheek.\n3. Grasp the far leg above the knee and pull it up, keeping the foot on the ground.\n4. Pull the knee toward you to roll them onto their side, facing you.\n5. Tilt the head back slightly to keep the airway open; check breathing continues.\n6. Stay with them, monitor breathing, and get emergency medical help.\nDo NOT put them in this position if you suspect a spinal injury — keep them still and get help instead, unless they are vomiting or you cannot maintain their airway any other way.", "content_sw": "Ikiwa mtu amepoteza fahamu lakini anapumua kawaida:\n1. Piga magoti karibu naye. Weka mkono ulio karibu nawe kwa pembe ya kulia kutoka mwilini, kiganja juu.\n2. Leta mkono wa mbali kupitia kifuani, mgongo wa mkono ukigusa shavu lililo karibu nawe.\n3. Shika mguu wa mbali juu ya goti na uuvute juu, ukiacha wayo ardhini.\n4. Vuta goti kuelekea kwako ili amzungushe ubavuni, akikutazama.\n5. Inamisha kichwa nyuma kidogo ili kufungua njia ya hewa; angalia anaendelea kupumua.\n6. Kaa naye, fuatilia kupumua kwake, na tafuta msaada wa dharura wa kimatibabu.\nUSIMWEKE katika mkao huu ikiwa unashuku jeraha la uti wa mgongo — mweke tuli na tafuta msaada badala yake, isipokuwa anatapika au huwezi kudumisha njia yake ya hewa kwa njia nyingine.", "source": "ICRC First Aid", "source_url": "https://www.icrc.org/en/first-aid", "region": "general", "verified": true}
{"id": "pepperspray-001", "domain": "first_aid", "title_en": "Pepper spray (OC) — differences from tear gas", "title_sw": "Dawa ya pilipili (OC) — tofauti na gesi ya machozi", "content_en": "Pepper spray (OC, oleoresin capsicum) is oil-based, unlike tear gas (CS/CN), so plain water alone is less effective.\n1. Move to fresh air immediately, upwind.\n2. Flush eyes with large amounts of cool water, or a diluted 1:1 baby shampoo and water rinse if available — this helps cut the oil.\n3. Do NOT rub eyes or skin — this drives the oil in deeper.\n4. Blot (do not rub) exposed skin with a cloth soaked in the same diluted shampoo solution, then rinse.\n5. Remove contaminated clothing carefully, avoiding contact with unaffected skin.\n6. Effects (burning, temporary blindness) usually ease within 30–60 minutes; seek medical help if breathing difficulty or symptoms persist beyond that.", "content_sw": "Dawa ya pilipili (OC) ina msingi wa mafuta, tofauti na gesi ya machozi (CS/CN), hivyo maji tu hayafanyi kazi vizuri.\n1. Sogea kwenye hewa safi mara moja, kuelekea upepo.\n2. Osha macho kwa maji mengi baridi, au mchanganyiko wa shampoo ya mtoto na maji (1:1) ikiwa inapatikana — hii husaidia kukata mafuta.\n3. USISUGUE macho au ngozi — hii inasukuma mafuta ndani zaidi.\n4. Gusa taratibu (usisugue) ngozi iliyoathirika kwa kitambaa chenye mchanganyiko huo wa shampoo, kisha osha.\n5. Vua nguo zilizochafuliwa kwa uangalifu, ukiepuka kugusa ngozi isiyoathirika.\n6. Athari (kuungua, upofu wa muda) kwa kawaida hupungua ndani ya dakika 30–60; tafuta msaada wa daktari ikiwa shida ya kupumua au dalili zinaendelea zaidi ya hapo.", "source": "EFF Surveillance Self-Defense / ICRC First Aid", "source_url": "https://ssd.eff.org/module/attending-protest", "region": "general", "verified": true}
{"id": "buddy-001", "domain": "evasion_safety", "title_en": "The buddy system — never go alone", "title_sw": "Mfumo wa rafiki — usiende peke yako", "content_en": "1. Always attend with at least one other person you trust — agree beforehand you will leave together.\n2. Exchange full names, a recent photo, and emergency contact details with your buddy before you go.\n3. Agree on a signal or phrase that means 'we need to leave now.'\n4. If you get separated, go to your pre-agreed rendezvous point rather than searching the crowd.\n5. Check in with someone outside the event by a set time — if they don't hear from you, they should start making calls.\n6. Watch out for each other's wellbeing, not just physical safety — stress and panic impair judgement.", "content_sw": "1. Hudhuria na angalau mtu mmoja unayemwamini — kubaliana mapema mtaondoka pamoja.\n2. Badilishana majina kamili, picha ya hivi karibuni, na maelezo ya mawasiliano ya dharura na rafiki yako kabla ya kwenda.\n3. Kubaliana juu ya ishara au maneno yanayomaanisha 'tunahitaji kuondoka sasa.'\n4. Mkitengana, nenda mahali mlipokubaliana kukutana badala ya kutafuta katika umati.\n5. Ripoti kwa mtu nje ya tukio kwa wakati uliokubaliwa — asipopata habari zako, aanze kupiga simu.\n6. Angalizeni ustawi wa kila mmoja, si tu usalama wa mwili — msongo na hofu huathiri uamuzi.", "source": "EFF Surveillance Self-Defense / Protection International", "source_url": "https://ssd.eff.org/module/attending-protest", "region": "general", "verified": true}
{"id": "exitroutes-001", "domain": "evasion_safety", "title_en": "Plan your exit routes before you arrive", "title_sw": "Panga njia zako za kutoka kabla ya kufika", "content_en": "1. When you arrive, identify at least TWO ways out that don't rely on the main entrance.\n2. Avoid narrow alleys, dead ends, and locked gates as planned exits — check they're actually open.\n3. Note landmarks (not just street names) so you can navigate even if signage or lighting is poor.\n4. Keep to the edges of large crowds where possible — the centre is hardest to exit quickly.\n5. If police or security forces begin to move in, leave via your planned route immediately — do not wait to see what happens.\n6. Share your planned route with your buddy or an out-of-area contact before you go.", "content_sw": "1. Ufikapo, tambua njia MBILI za kutoka ambazo hazitegemei mlango mkuu.\n2. Epuka vichochoro vyembamba, njia zisizo na mwisho, na malango yaliyofungwa kama njia za kutoka — hakikisha ni wazi.\n3. Angalia alama za mahali (si tu majina ya barabara) ili uweze kuelekea hata ikiwa alama au mwanga hafifu.\n4. Kaa pembezoni mwa umati mkubwa inapowezekana — katikati ni ngumu zaidi kutoka haraka.\n5. Polisi au vikosi vya usalama vikianza kusonga, ondoka kupitia njia yako iliyopangwa mara moja — usisubiri kuona kitakachotokea.\n6. Shiriki njia yako iliyopangwa na rafiki yako au mtu wa nje ya eneo kabla ya kwenda.", "source": "EFF Surveillance Self-Defense", "source_url": "https://ssd.eff.org/module/attending-protest", "region": "general", "verified": true}
{"id": "curfew-001", "domain": "evasion_safety", "title_en": "Curfew safety", "title_sw": "Usalama wakati wa amri ya kutotoka nje", "content_en": "If a curfew is announced:\n1. Confirm the exact hours and area from an official government source or major news outlet — rumours spread fast and are often wrong.\n2. Plan to be at your destination well before the curfew start time — allow extra time for traffic and crowds.\n3. If you must move during curfew for a genuine emergency, carry identification and be prepared to explain your reason calmly.\n4. Avoid unnecessary movement — most risk during curfews comes from unclear, chaotic encounters after dark.\n5. Know that under Article 24 of the Constitution, any curfew restricting movement must be reasonable, justifiable, and proportionate — note the legal basis cited if you can, for later reference with LSK or KHRC.", "content_sw": "Amri ya kutotoka nje ikitangazwa:\n1. Thibitisha masaa na eneo halisi kutoka chanzo rasmi cha serikali au vyombo vikuu vya habari — uvumi huenea haraka na mara nyingi ni wa uongo.\n2. Panga kuwa mahali unapoenda kabla ya muda wa kuanza kwa amri — ruhusu muda wa ziada kwa msongamano wa magari na watu.\n3. Ikiwa lazima usogee wakati wa amri kwa dharura halisi, beba kitambulisho na uwe tayari kueleza sababu yako kwa utulivu.\n4. Epuka mwendo usio wa lazima — hatari nyingi wakati wa amri hutokana na mikutano isiyo wazi, ya fujo baada ya giza.\n5. Jua kwamba chini ya Kifungu cha 24 cha Katiba, amri yoyote inayozuia mwendo lazima iwe ya busara, inayoweza kuhesabiwa, na ya uwiano — andika msingi wa kisheria uliotajwa kwa rejeleo la baadaye na LSK au KHRC.", "source": "Constitution of Kenya 2010, Article 24 / LSK", "source_url": "https://www.lsk.or.ke", "region": "kenya", "verified": true}
{"id": "radio-001", "domain": "blackout_comms", "title_en": "FM radio for information during a network blackout", "title_sw": "Redio ya FM kwa habari wakati wa kuzimwa kwa mtandao", "content_en": "1. A battery-powered or wind-up FM radio works without any mobile network or internet.\n2. Tune to known, established national or local stations for verified updates rather than unverified rumours.\n3. Many basic phones and some smartphones have a built-in FM radio chip — check Settings for an 'FM Radio' app (usually needs wired headphones as the antenna).\n4. Keep a small battery-powered radio and spare batteries in your emergency kit at home.\n5. Radio is one-way — it won't let you contact anyone — but it is highly reliable for situational awareness when data networks are down.", "content_sw": "1. Redio ya FM inayotumia betri au ya kujikaza inafanya kazi bila mtandao wowote wa simu au intaneti.\n2. Sikiliza vituo vinavyojulikana na vilivyoimarika vya kitaifa au vya mtaa kwa habari zilizothibitishwa badala ya uvumi.\n3. Simu nyingi za kawaida na baadhi ya simu janja zina chip ya redio ya FM iliyojengwa ndani — angalia Mipangilio kwa programu ya 'FM Radio' (kwa kawaida inahitaji hedfoni zenye waya kama antena).\n4. Weka redio ndogo inayotumia betri na betri za akiba kwenye kifurushi chako cha dharura nyumbani.\n5. Redio ni njia moja tu — haitakuruhusu kuwasiliana na mtu yeyote — lakini ni ya kuaminika sana kwa uelewa wa hali wakati mitandao ya data imezimwa.", "source": "EFF Surveillance Self-Defense / Access Now", "source_url": "https://www.accessnow.org/help", "region": "general", "verified": true}
{"id": "vpn-001", "domain": "blackout_comms", "title_en": "VPNs when the internet is throttled or restricted", "title_sw": "VPN wakati intaneti imepunguzwa au kuzuiwa", "content_en": "1. A VPN (Virtual Private Network) can sometimes restore access when specific sites/apps are blocked, but it does NOT work if mobile data or Wi-Fi is fully cut off at the network level.\n2. Install a reputable VPN app BEFORE you need it — you cannot download one once access is blocked.\n3. Free VPNs can log and sell your data — prefer well-reviewed providers with a clear no-logs policy where possible.\n4. A VPN does not make you anonymous to someone who already has physical access to your device.\n5. If a full shutdown is in effect, no VPN can restore connectivity — plan for offline tools (Briar, FM radio, pre-agreed rendezvous) as your primary blackout plan, and treat VPN as a secondary option for throttling only.", "content_sw": "1. VPN (Virtual Private Network) wakati mwingine inaweza kurejesha ufikiaji wakati tovuti/programu maalum zimezuiwa, lakini HAIFANYI kazi ikiwa data ya simu au Wi-Fi imezimwa kabisa katika kiwango cha mtandao.\n2. Sakinisha programu ya VPN inayoaminika KABLA hujaihitaji — huwezi kupakua moja mtandao ukishazuiwa.\n3. VPN za bure zinaweza kuhifadhi na kuuza data yako — pendelea watoa huduma wenye sifa nzuri na sera wazi ya kutohifadhi kumbukumbu inapowezekana.\n4. VPN haikufanyi usijulikane kwa mtu ambaye tayari ana ufikiaji wa moja kwa moja wa kifaa chako.\n5. Ikiwa kuzimwa kamili kunatumika, hakuna VPN inayoweza kurejesha muunganisho — panga zana za nje ya mtandao (Briar, redio ya FM, mahali pa kukutana palipokubaliwa) kama mpango wako mkuu, na VPN kama chaguo la pili kwa kupunguzwa tu.", "source": "Access Now Digital Security Helpline / EFF SSD", "source_url": "https://www.accessnow.org/help", "region": "general", "verified": true}
{"id": "photo-001", "domain": "documentation", "title_en": "Photographing safely without endangering others", "title_sw": "Kupiga picha kwa usalama bila kuhatarisha wengine", "content_en": "1. Before sharing, blur or crop out the faces of protesters, bystanders, and anyone who did not consent to be identified — a face in a photo can be used to identify and target someone later.\n2. Do not photograph people's identifying details (name badges, home addresses, car plates of bystanders) unless documenting misconduct where this is the point.\n3. Ask yourself who could be harmed if this photo is seen by the wrong person, including facial recognition tools.\n4. When documenting officers or misconduct, focus on badge numbers, uniforms, vehicles — the accountability target — not on other protesters.\n5. Turn off automatic cloud backup for photos taken at sensitive events, or use an app with a locked, hidden gallery, until you can review and secure them.", "content_sw": "1. Kabla ya kushiriki, fifisha au kata nyuso za waandamanaji, watazamaji, na mtu yeyote ambaye hakukubali kutambuliwa — uso kwenye picha unaweza kutumika kutambua na kulenga mtu baadaye.\n2. Usipige picha za maelezo ya utambulisho wa watu (beji za majina, anwani za nyumbani, nambari za bati za watazamaji) isipokuwa unadokumenta uovu ambapo hii ndiyo lengo.\n3. Jiulize ni nani anayeweza kudhurika ikiwa picha hii itaonwa na mtu mbaya, ikiwa ni pamoja na zana za utambuzi wa uso.\n4. Unapodokumenta maafisa au uovu, zingatia nambari za beji, sare, magari — lengo la uwajibikaji — si waandamanaji wengine.\n5. Zima uhifadhi wa kiotomatiki wa wingu kwa picha zilizopigwa kwenye matukio nyeti, au tumia programu yenye picha zilizofichwa na kufungwa, hadi uweze kuzikagua na kuzihifadhi salama.", "source": "EFF Surveillance Self-Defense", "source_url": "https://ssd.eff.org/module/attending-protest", "region": "general", "verified": true}
{"id": "metadata-001", "domain": "documentation", "title_en": "Strip metadata before sharing photos or videos", "title_sw": "Ondoa metadata kabla ya kushiriki picha au video", "content_en": "Photos and videos often contain hidden metadata (EXIF data): exact GPS location, device model, and timestamp — this can reveal where you or a source were standing.\n1. On Android: many messaging apps (WhatsApp, Signal) automatically strip most metadata when sending — but check before relying on this.\n2. Turn off location tagging in your camera app settings BEFORE the event: Camera app > Settings > Location tags, off.\n3. If sending directly as a file (not via a messaging app), use a metadata-removal app, or screenshot the image (screenshots typically do not carry the original EXIF data).\n4. When sending to a journalist or human rights organisation for investigation, metadata can be valuable evidence — in that case, keep the original and only strip metadata from copies meant for public sharing.", "content_sw": "Picha na video mara nyingi zina metadata iliyofichwa (data ya EXIF): mahali halisi pa GPS, aina ya kifaa, na wakati — hii inaweza kufichua mahali wewe au chanzo mlisimama.\n1. Kwenye Android: programu nyingi za ujumbe (WhatsApp, Signal) huondoa metadata nyingi kiotomatiki zinapotuma — lakini hakikisha kabla ya kutegemea hii.\n2. Zima uwekaji lebo za mahali kwenye mipangilio ya programu ya kamera KABLA ya tukio: Programu ya Kamera > Mipangilio > Lebo za mahali, zima.\n3. Ikiwa unatuma moja kwa moja kama faili (si kupitia programu ya ujumbe), tumia programu ya kuondoa metadata, au piga picha ya skrini (screenshot mara nyingi haibebi data asili ya EXIF).\n4. Unapotuma kwa mwandishi wa habari au shirika la haki za binadamu kwa uchunguzi, metadata inaweza kuwa ushahidi wa thamani — katika hali hiyo, hifadhi ya asili na uondoe metadata tu kwa nakala za kushiriki hadharani.", "source": "EFF Surveillance Self-Defense", "source_url": "https://ssd.eff.org/module/your-security-plan", "region": "general", "verified": true}
{"id": "evidence-001", "domain": "documentation", "title_en": "Encrypting and backing up evidence securely", "title_sw": "Kusimbua na kuhifadhi nakala ya ushahidi kwa usalama", "content_en": "1. Back up sensitive photos/videos to a trusted person's device or an encrypted cloud service as soon as you have Wi-Fi — do not keep the only copy on the phone that was at the scene.\n2. Use an app with built-in encrypted backup, or manually copy files over a cable to a computer with full-disk encryption enabled.\n3. Label files with date, approximate time, and location — but store this index separately from the files if the content itself is highly sensitive.\n4. Consider sharing a copy with a human rights organisation (Missing Voices, KHRC, IMLU) — this creates an independent record even if your own copy is lost or seized.\n5. Delete sensitive material from the original device once safely backed up, if carrying it further poses risk to you.", "content_sw": "1. Hifadhi nakala za picha/video nyeti kwenye kifaa cha mtu unayemwamini au huduma ya wingu iliyosimbwa mara tu unapopata Wi-Fi — usiache nakala pekee kwenye simu iliyokuwa eneo la tukio.\n2. Tumia programu yenye uhifadhi wa nakala uliosimbwa, au nakili faili kwa kebo kwenye kompyuta yenye usimbuaji kamili wa diski.\n3. Weka lebo za faili kwa tarehe, muda wa karibu, na mahali — lakini hifadhi orodha hii tofauti na faili ikiwa maudhui yenyewe ni nyeti sana.\n4. Fikiria kushiriki nakala na shirika la haki za binadamu (Missing Voices, KHRC, IMLU) — hii inaunda rekodi huru hata kama nakala yako itapotea au kunyakuliwa.\n5. Futa maudhui nyeti kutoka kifaa asili mara tu yamehifadhiwa salama, ikiwa kuyabeba zaidi kunakuletea hatari.", "source": "EFF Surveillance Self-Defense / Access Now", "source_url": "https://ssd.eff.org/module/your-security-plan", "region": "general", "verified": true}
{"id": "press-001", "domain": "rights", "title_en": "Rights of journalists and media at protests", "title_sw": "Haki za waandishi wa habari na vyombo vya habari maandamanoni", "content_en": "Article 34 of the Constitution of Kenya 2010 guarantees freedom of the media, including freedom to seek, receive, and impart information.\n1. Journalists have the right to observe, film, and report on public assemblies.\n2. Press credentials should be visible but officers cannot lawfully confiscate equipment without due legal process.\n3. If equipment is seized or damaged, document the officer's name/badge number if possible and report to LSK, KHRC, or a journalists' association.\n4. Non-journalists documenting events (citizen journalism) are also protected under Article 33 (freedom of expression) and Article 35 (access to information), though press credentials may afford practical recognition at the scene.\n5. Report any assault or obstruction of media work to KHRC or LSK.", "content_sw": "Kifungu cha 34 cha Katiba ya Kenya 2010 kinahakikisha uhuru wa vyombo vya habari, ikiwa ni pamoja na uhuru wa kutafuta, kupokea, na kusambaza habari.\n1. Waandishi wa habari wana haki ya kutazama, kupiga picha, na kuripoti mikutano ya umma.\n2. Vitambulisho vya habari vinapaswa kuonekana lakini maafisa hawawezi kunyakua vifaa kihalali bila mchakato wa kisheria ufaao.\n3. Vifaa vikinyakuliwa au kuharibiwa, dokumenta jina/nambari ya beji ya afisa ikiwezekana na uripoti kwa LSK, KHRC, au chama cha waandishi wa habari.\n4. Wasio waandishi wa habari wanaodokumenta matukio (uandishi wa habari wa raia) pia wanalindwa chini ya Kifungu cha 33 (uhuru wa kujieleza) na Kifungu cha 35 (upatikanaji wa habari), ingawa vitambulisho vya habari vinaweza kutoa utambuzi wa vitendo eneo la tukio.\n5. Ripoti unyanyasaji wowote au uzuiaji wa kazi ya vyombo vya habari kwa KHRC au LSK.", "source": "Constitution of Kenya 2010, Articles 33-35 / LSK", "source_url": "https://www.lsk.or.ke", "region": "kenya", "verified": true}
{"id": "recordpolice-001", "domain": "rights", "title_en": "Your right to record police in public", "title_sw": "Haki yako ya kurekodi polisi hadharani", "content_en": "1. Recording police performing their public duties in a public place is a lawful exercise of your rights under Article 33 (freedom of expression) and Article 35 (access to information) of the Constitution of Kenya 2010.\n2. Stand at a reasonable distance so you are not accused of obstruction.\n3. You do not have to stop recording simply because an officer tells you to — but weigh your immediate safety; de-escalation may matter more in the moment than asserting the point.\n4. Officers cannot lawfully delete your footage or confiscate your phone without due legal process.\n5. If your phone is seized unlawfully, note the officer's name/badge number, time, and location, and report to LSK or KHRC as soon as you can.", "content_sw": "1. Kurekodi polisi wakifanya kazi zao za umma mahali pa umma ni zoezi halali la haki zako chini ya Kifungu cha 33 (uhuru wa kujieleza) na Kifungu cha 35 (upatikanaji wa habari) cha Katiba ya Kenya 2010.\n2. Simama umbali unaofaa ili usishtakiwe kwa kuzuia.\n3. Sio lazima uache kurekodi kwa sababu tu afisa amekuambia — lakini pima usalama wako wa haraka; kupunguza msongo kunaweza kuwa muhimu zaidi wakati huo kuliko kusisitiza jambo hilo.\n4. Maafisa hawawezi kihalali kufuta rekodi yako au kunyakua simu yako bila mchakato wa kisheria ufaao.\n5. Simu yako ikinyakuliwa kinyume cha sheria, andika jina/nambari ya beji ya afisa, wakati, na mahali, na uripoti kwa LSK au KHRC haraka iwezekanavyo.", "source": "Constitution of Kenya 2010, Articles 33 & 35 / LSK", "source_url": "https://www.lsk.or.ke", "region": "kenya", "verified": true}
{"id": "minors-001", "domain": "rights", "title_en": "If a minor (under 18) is arrested or detained", "title_sw": "Ikiwa mtoto (chini ya miaka 18) amekamatwa au kuzuiliwa", "content_en": "Article 53 of the Constitution of Kenya 2010 protects children, stating detention should be a measure of last resort and for the shortest appropriate period.\n1. A parent, guardian, or an appropriate adult should be informed and present as soon as possible.\n2. The child must be kept separate from adult detainees.\n3. The child has the same core rights as any detainee: to be informed of the reason for arrest, to remain silent, and to legal representation (Article 49).\n4. Contact LSK immediately for legal assistance specific to minors — note this is a priority case.\n5. Document the child's full name, age, time and location of arrest, and any officer details, as soon as it is safe.\n6. Report to Missing Voices or KHRC if the child's whereabouts become unknown.", "content_sw": "Kifungu cha 53 cha Katiba ya Kenya 2010 kinalinda watoto, kikieleza kuwa kizuizi kinapaswa kuwa hatua ya mwisho na kwa muda mfupi zaidi unaofaa.\n1. Mzazi, mlezi, au mtu mzima anayefaa anapaswa kujulishwa na kuwepo haraka iwezekanavyo.\n2. Mtoto lazima awekwe mbali na wafungwa wazima.\n3. Mtoto ana haki sawa za msingi kama mfungwa yeyote: kujulishwa sababu ya kukamatwa, kukaa kimya, na uwakilishi wa kisheria (Kifungu cha 49).\n4. Wasiliana na LSK mara moja kwa msaada wa kisheria maalum kwa watoto — hii ni kesi ya kipaumbele.\n5. Andika jina kamili la mtoto, umri, wakati na mahali pa kukamatwa, na maelezo yoyote ya afisa, mara tu inapokuwa salama.\n6. Ripoti kwa Missing Voices au KHRC ikiwa mahali alipo mtoto hapajulikani.", "source": "Constitution of Kenya 2010, Articles 49 & 53 / LSK", "source_url": "https://www.lsk.or.ke", "region": "kenya", "verified": true}
{"id": "contacts-lsk-001", "domain": "contacts", "title_en": "Legal aid — Law Society of Kenya (LSK)", "title_sw": "Msaada wa kisheria — Jumuiya ya Sheria Kenya (LSK)", "content_en": "LSK (Law Society of Kenya) provides legal referrals, detainee tracking assistance, and information on free legal aid.\nPHONE: +254 799 595 800\nWEBSITE: lsk.or.ke\nUse this contact if you or someone you know has been arrested and needs a lawyer, or to ask 'How to access Free Legal Aid?' via their website.\nNOTE: Verify this number on the official site before relying on it — numbers can change.", "content_sw": "LSK (Jumuiya ya Sheria Kenya) inatoa rufaa za kisheria, msaada wa kufuatilia wafungwa, na maelezo ya msaada wa kisheria wa bure.\nSIMU: +254 799 595 800\nTOVUTI: lsk.or.ke\nTumia mawasiliano haya ikiwa wewe au mtu unayemjua amekamatwa na anahitaji wakili, au kuuliza 'How to access Free Legal Aid?' kupitia tovuti yao.\nNOTA: Thibitisha nambari hii kwenye tovuti rasmi kabla ya kuitegemea — nambari zinaweza kubadilika.", "source": "Law Society of Kenya", "source_url": "https://www.lsk.or.ke", "region": "kenya", "verified": true}
{"id": "contacts-imlu-001", "domain": "contacts", "title_en": "Torture and ill-treatment documentation — IMLU", "title_sw": "Kudokumenta mateso na ukatili — IMLU", "content_en": "IMLU (Independent Medico-Legal Unit) documents torture, ill-treatment, and extrajudicial killings, and can connect survivors with medical and legal support.\nTOLL-FREE: 0800 721 401\nWEBSITE: imlu.org\nUse this contact if you or someone you know has experienced torture or serious ill-treatment while in custody.\nNOTE: Verify this number on the official site before relying on it — numbers can change.", "content_sw": "IMLU (Kitengo Huru cha Kimatibabu na Kisheria) inadokumenta mateso, ukatili, na mauaji yasiyo halali, na inaweza kuunganisha manusura na msaada wa kimatibabu na kisheria.\nBILA MALIPO: 0800 721 401\nTOVUTI: imlu.org\nTumia mawasiliano haya ikiwa wewe au mtu unayemjua amepitia mateso au ukatili mkubwa akiwa kizuizini.\nNOTA: Thibitisha nambari hii kwenye tovuti rasmi kabla ya kuitegemea — nambari zinaweza kubadilika.", "source": "Independent Medico-Legal Unit (IMLU)", "source_url": "https://www.imlu.org", "region": "kenya", "verified": true}
{"id": "contacts-missingvoices-001", "domain": "contacts", "title_en": "Reporting a disappearance — Missing Voices", "title_sw": "Kuripoti kutoweka — Missing Voices", "content_en": "Missing Voices is a coalition documenting enforced disappearances and extrajudicial killings in Kenya, run by KHRC, IMLU, Amnesty Kenya, and other partners.\nWEBSITE: missingvoices.or.ke\nREPORT: use the 'Report Incident' page on their website to file a documented record.\nNOTE: Missing Voices operates primarily through its website reporting form rather than a phone hotline — for urgent, time-critical help finding a missing person, also contact Kenya Red Cross (1199) or go directly to a police station for an OB (Occurrence Book) number.", "content_sw": "Missing Voices ni muungano unaodokumenta kutoweka bila haki na mauaji yasiyo halali Kenya, unaoendeshwa na KHRC, IMLU, Amnesty Kenya, na washirika wengine.\nTOVUTI: missingvoices.or.ke\nRIPOTI: tumia ukurasa wa 'Report Incident' kwenye tovuti yao kuweka rekodi iliyodokumentwa.\nNOTA: Missing Voices hufanya kazi hasa kupitia fomu ya kuripoti ya tovuti yao badala ya nambari ya simu — kwa msaada wa haraka, wa dharura wa kutafuta mtu aliyepotea, pia wasiliana na Kenya Red Cross (1199) au nenda moja kwa moja kituo cha polisi kupata nambari ya OB (Occurrence Book).", "source": "Missing Voices Kenya", "source_url": "https://missingvoices.or.ke", "region": "kenya", "verified": true}
{"id": "eyeinjury-001", "domain": "first_aid", "title_en": "Eye injury from a projectile (rubber bullet, shrapnel, stone)", "title_sw": "Jeraha la jicho kutoka kwa kitu kilichorushwa (risasi ya mpira, kipande, jiwe)", "content_en": "A direct hit to the eye is a serious emergency — the goal is to prevent further damage before reaching help.\n1. Do NOT press on, rub, or try to rinse an eye with an embedded object or visible open injury.\n2. Do NOT attempt to remove anything stuck in or near the eye.\n3. Cover BOTH eyes lightly (covering only the injured eye still causes the injured eye to move in sync with the other) with a clean cloth or eye shield — do not apply pressure.\n4. Keep the person calm, sitting or lying with the head still.\n5. Do not give food or drink in case surgery is needed soon.\n6. Get to emergency medical care immediately — eye injuries can be time-critical for saving vision.\nFor chemical irritation without an embedded object or open wound, see the tear gas / pepper spray cards instead.", "content_sw": "Kupigwa moja kwa moja kwa jicho ni dharura kubwa — lengo ni kuzuia uharibifu zaidi kabla ya kufika msaada.\n1. USIBONYEZE, usisugue, au ujaribu kuosha jicho lenye kitu kilichopachikwa au jeraha wazi linaloonekana.\n2. USIJARIBU kuondoa kitu chochote kilichopachikwa ndani au karibu na jicho.\n3. Funika MACHO YOTE MAWILI kwa upole (kufunika jicho lililoumia tu bado linasababisha jicho hilo kusogea sawia na lingine) kwa kitambaa safi au kinga ya jicho — usiweke shinikizo.\n4. Mweke mtu tulivu, ameketi au amelala na kichwa kikiwa tuli.\n5. Usimpe chakula wala kinywaji endapo upasuaji utahitajika hivi karibuni.\n6. Tafuta huduma ya dharura ya kimatibabu mara moja — majeraha ya jicho yanaweza kuwa ya hatari ya wakati kwa kuokoa uwezo wa kuona.\nKwa muwasho wa kemikali bila kitu kilichopachikwa au jeraha wazi, angalia kadi za gesi ya machozi / dawa ya pilipili badala yake.", "source": "ICRC First Aid", "source_url": "https://www.icrc.org/en/first-aid", "region": "general", "verified": true}
{"id": "dualsim-001", "domain": "blackout_comms", "title_en": "Prepaid SIM and dual-SIM backup plan", "title_sw": "Mpango wa akiba wa SIM ya kulipia mapema na simu ya SIM mbili", "content_en": "1. If your phone supports dual SIM, carry a second SIM from a different network provider — if one network is throttled or down, the other may still work.\n2. Buy a spare prepaid SIM in advance and keep it topped up with a small airtime balance, stored separately from your main phone.\n3. Register any spare SIM under your own name where required — an unregistered or improperly acquired SIM can itself become a legal complication.\n4. Save your emergency contacts' numbers to the SIM card itself (not just cloud-synced contacts) so they survive a factory reset or phone swap.\n5. Know your own number by heart — write it on paper too, in case you need to give it to someone helping you.", "content_sw": "1. Ikiwa simu yako inaunga mkono SIM mbili, beba SIM ya pili kutoka kampuni tofauti ya mtandao — mtandao mmoja ukipunguzwa au kuzimwa, mwingine unaweza kuendelea kufanya kazi.\n2. Nunua SIM ya akiba ya kulipia mapema mapema na uiweke na salio dogo la muda wa maongezi, ikihifadhiwa tofauti na simu yako kuu.\n3. Sajili SIM yoyote ya akiba kwa jina lako popote inapohitajika — SIM isiyosajiliwa au iliyopatikana isivyo sahihi inaweza yenyewe kuwa tatizo la kisheria.\n4. Hifadhi nambari za mawasiliano yako ya dharura kwenye kadi ya SIM yenyewe (si tu anwani zilizosawazishwa na wingu) ili zisalie hata baada ya kuweka upya kiwandani au kubadilisha simu.\n5. Jua nambari yako mwenyewe kwa moyo — iandike kwa karatasi pia, endapo utahitaji kumpa mtu anayekusaidia.", "source": "EFF Surveillance Self-Defense / Access Now", "source_url": "https://ssd.eff.org/module/your-security-plan", "region": "general", "verified": true}
"""


In [3]:
import json

cards = [json.loads(line) for line in CARDS_JSONL.strip().split("\n") if line.strip()]
cards = [c for c in cards if c["verified"]]  # rag.dart excludes unverified cards from retrieval

by_domain = {}
for c in cards:
    by_domain.setdefault(c["domain"], []).append(c["id"])

print(f"Loaded {len(cards)} verified cards\n")
for domain, ids in sorted(by_domain.items()):
    print(f"  {domain:18s} {len(ids):2d}  {ids}")


Loaded 41 verified cards

  blackout_comms      7  ['comms-001', 'comms-002', 'comms-003', 'comms-004', 'radio-001', 'vpn-001', 'dualsim-001']
  contacts            4  ['contacts-001', 'contacts-lsk-001', 'contacts-imlu-001', 'contacts-missingvoices-001']
  documentation       6  ['doc-001', 'doc-002', 'phone-001', 'photo-001', 'metadata-001', 'evidence-001']
  evasion_safety      7  ['evasion-001', 'evasion-002', 'evasion-003', 'evasion-004', 'buddy-001', 'exitroutes-001', 'curfew-001']
  first_aid          11  ['teargas-001', 'teargas-002', 'firstaid-001', 'firstaid-002', 'firstaid-003', 'firstaid-004', 'choking-001', 'heat-001', 'unconscious-001', 'pepperspray-001', 'eyeinjury-001']
  rights              6  ['rights-001', 'rights-002', 'rights-003', 'press-001', 'recordpolice-001', 'minors-001']


## 2. Retrieval — ports `lib/services/rag.dart`

Simple keyword + prefix-match scoring over the corpus, with a deterministic keyword→domain router as a fast path
(the same two-stage design as the app: try the detected domain first, fall back to the full corpus if nothing
scores above threshold). This is a direct line-for-line port of `RagService` in `rag.dart` — same stop words, same
scoring formula (`matches / query_token_count`), same default `k=4` and `threshold=0.05`, same domain keyword
lists.

In [4]:
STOP_WORDS = {
    "the", "a", "an", "is", "are", "was", "were", "be", "been", "being",
    "have", "has", "had", "do", "does", "did", "will", "would", "could",
    "should", "may", "might", "can", "shall", "to", "of", "in", "on",
    "at", "by", "for", "with", "about", "from", "that", "this", "it",
    "its", "my", "your", "our", "their", "if", "and", "or", "but", "not",
    "no", "so", "up", "out", "what", "how", "when", "where", "who", "which",
    "na", "ya", "wa", "kwa", "ni", "la", "za", "au", "si", "ikiwa", "hii",
    "hilo", "hiyo", "yake", "wake", "wao", "mimi", "wewe", "sisi", "nyinyi",
}

import re

def tokenise(text):
    text = re.sub(r"[^\w\s]", " ", text)
    return [t for t in text.split() if len(t) > 2 and t not in STOP_WORDS]


def title(card, lang):
    return card["title_sw"] if lang in ("sw", "sheng") else card["title_en"]


def content(card, lang):
    return card["content_sw"] if lang in ("sw", "sheng") else card["content_en"]


def score(query_tokens, card, lang):
    card_text = f"{title(card, lang)} {content(card, lang)} {card['domain']}".lower()
    card_tokens = tokenise(card_text)
    if not card_tokens:
        return 0.0
    matches = 0
    for qt in query_tokens:
        if qt in card_tokens:
            matches += 1
            continue
        if any(ct.startswith(qt) or qt.startswith(ct) for ct in card_tokens):
            matches += 1
    return matches / len(query_tokens)


DOMAIN_KEYWORDS = {
    "first_aid": ["tear gas", "machozi", "gas", "eyes", "macho", "inhale", "burn", "rubber bullet",
                  "baton", "struck", "first aid", "damu", "bleeding", "shock", "mshtuko", "jeraha"],
    "evasion_safety": ["follow", "fuatwa", "followed", "checkpoint", "kizuizi", "escape", "hide",
                        "crowd", "crush", "stampede", "msongamano", "run"],
    "blackout_comms": ["network", "mtandao", "internet", "offline", "blackout", "reach", "fikia",
                        "contact", "signal", "briar", "rendezvous", "kukutana", "wifi", "message", "call"],
    "documentation": ["witness", "shuhudia", "record", "rekodi", "abduction", "utekaji", "report",
                       "ripoti", "document", "disappeared", "grabbed", "kamatwa", "arrested",
                       "phone", "simu", "prepare", "andaa", "encryption", "pin", "surveillance", "monitored"],
    "rights": ["rights", "haki", "detained", "lawyer", "protest right", "constitution", "katiba", "silent"],
    "contacts": ["emergency", "hospital", "dharura", "number", "nambari", "hotline"],
}


def detect_domain(query):
    q = query.lower()
    for domain, keywords in DOMAIN_KEYWORDS.items():
        if any(kw in q for kw in keywords):
            return domain
    return None


def retrieve_top_k(query, lang="en", domain=None, k=4, threshold=0.05):
    query_tokens = tokenise(query.lower())
    if not query_tokens:
        return []

    detected_domain = domain or detect_domain(query)

    scored = []
    for card in cards:
        if detected_domain and card["domain"] != detected_domain:
            continue
        s = score(query_tokens, card, lang)
        if s >= threshold:
            scored.append((s, card))

    if not scored and detected_domain:
        for card in cards:
            s = score(query_tokens, card, lang)
            if s >= threshold:
                scored.append((s, card))

    scored.sort(key=lambda x: -x[0])
    return [c for _, c in scored[:k]]


def format_for_prompt(retrieved, lang):
    if not retrieved:
        return "(No relevant cards found in corpus.)"
    parts = []
    for i, c in enumerate(retrieved, 1):
        parts.append(f"[{i}] {title(c, lang)} ({c['source']})\n{content(c, lang)}\n")
    return "\n".join(parts)


# Quick sanity check
_test = retrieve_top_k("There is tear gas. What do I do right now?", lang="en")
print(f"Retrieved {len(_test)} cards for tear gas query: {[c['id'] for c in _test]}")


Retrieved 4 cards for tear gas query: ['teargas-001', 'teargas-002', 'pepperspray-001', 'eyeinjury-001']


## 3. Prompts — ports `lib/prompts/prompts.dart`

The three prompt templates below are a direct port of `Prompts` in `prompts.dart` — same rules, same refusal
string, same disclaimer, same language instructions for English / Kiswahili / Sheng, same JSON extraction schema.
Nothing here was rewritten or "improved" for the notebook; it is the app's actual prompt text.

In [5]:
def language_instruction(lang):
    if lang == "sw":
        return "Reply in Kiswahili. Use clear, simple Kiswahili that any Kenyan can understand."
    if lang == "sheng":
        return ("Reply in Sheng — the Nairobi street mix of Kiswahili and English. "
                "Keep it natural and conversational, as a Nairobi young person would speak. "
                "Mix Sheng vocabulary (e.g. sema, fanya, gari, fiti, mbaya) with clear Kiswahili for safety terms.")
    return "Reply in English."


def grounded_answer_prompt(query, retrieved_cards, language):
    lang_instruction = language_instruction(language)
    return f"""You are Niko Rada, an offline safety companion built to help people in Kenya during protests, network blackouts, and emergencies.

RULES (non-negotiable):
1. Answer ONLY using the SOURCES provided below. Do not use any external knowledge.
2. If the SOURCES do not cover the question, say exactly: "Sina taarifa ya hilo katika maktaba yangu / I don't have that in my library. Please consult a doctor, lawyer, or trusted authority." Do not guess or invent.
3. Be CALM, CONCRETE, and BRIEF. Use numbered steps. Most important action first.
4. {lang_instruction}
5. End every answer with this disclaimer on its own line: "_Mwongozo tu, si dhamana. Si ushauri wa kisheria au kimatibabu. / Guidance only, not a guarantee. Not legal or medical advice._"
6. Cite the source of your answer using [Source Name] at the end of your answer, before the disclaimer.

SOURCES:
{retrieved_cards}

USER SITUATION: {query}
"""


def extract_incident_prompt(free_text):
    return f"""Extract a structured incident report from the account below.
Output ONLY valid JSON with exactly these fields:
{{
  "who": "description of person(s) taken or involved",
  "what": "what happened",
  "where": "location — street, landmark, area",
  "when": "time and date if mentioned",
  "details": "any additional details — vehicle, clothing, uniforms, what was said",
  "witnesses": "any witnesses mentioned",
  "source_reliability": "firsthand / secondhand / unknown"
}}
Use "unknown" for any field that is not mentioned. Do NOT invent details.
Output ONLY the JSON object. No explanation, no markdown, no extra text.

ACCOUNT: {free_text}
"""

print("Prompt templates loaded (ports of prompts.dart).")


Prompt templates loaded (ports of prompts.dart).


## 4. Loading Gemma 4 E2B

The app targets `litert-community/gemma-4-E2B-it-litert-lm` (the `.litertlm` build, loaded via
`flutter_gemma_litertlm`'s LiteRT-LM FFI engine — see `lib/services/gemma_service.dart`). That native engine
doesn't run inside a Jupyter kernel, so here we load the **same Gemma 4 E2B model family** via `transformers`
instead: [`google/gemma-4-E2B-it`](https://huggingface.co/google/gemma-4-E2B-it), open access (Apache 2.0, no
gating).

Gemma 4 E2B is natively multimodal (`AutoModelForMultimodalLM` + `AutoProcessor`) — we only use its text path here,
matching how the app uses it (text-only prompts, no image/audio input). If the installed `transformers` version on
this Kaggle image is too old to recognise the brand-new multimodal model class, this cell falls back to
[`google/gemma-3-4b-it`](https://huggingface.co/google/gemma-3-4b-it) (same Gemma family, long-established
`transformers` support) so the notebook still runs top-to-bottom rather than failing on an environment mismatch.
The fallback is logged clearly if it triggers — the retrieval, prompts, and demo logic below are 100% identical
either way; only the model backend changes.

In [6]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    if any(f.endswith(('.safetensors', '.gguf', '.bin')) for f in files):
        print(root)

/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1


In [7]:
import torch, os
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_PATH = "/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1"
assert os.path.exists(MODEL_PATH), f"Not found: {MODEL_PATH}"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, device_map="auto", quantization_config=bnb,
)
MODEL_BACKEND = f"Gemma 4 E2B-it (local Kaggle input, 4-bit)"
print("Loaded:", MODEL_BACKEND)

GEMMA_EOT = tokenizer.convert_tokens_to_ids("<end_of_turn>")

print("chat_template present:", tokenizer.chat_template is not None)
EOT = tokenizer.convert_tokens_to_ids("<end_of_turn>")
print("EOT id:", EOT, "| eos id:", tokenizer.eos_token_id)

import re as _re

def generate(prompt, max_new_tokens=800):
    if tokenizer.chat_template:
        inputs = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            add_generation_prompt=True, tokenize=True,
            return_dict=True, return_tensors="pt",
        ).to(model.device)
    else:
        text = f"<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n"
        inputs = tokenizer(text, return_tensors="pt").to(model.device)

    stops = [t for t in {tokenizer.eos_token_id, EOT} if t is not None]
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            repetition_penalty=1.15, eos_token_id=stops,
        )
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

    # strip leaked thinking block and role prefixes
    text = _re.sub(r"^\s*thought\b.*?(?=\n\s*\n|\Z)", "", text, flags=_re.S | _re.I).strip()
    text = _re.sub(r"^(model\s*)+", "", text).strip()
    return text

print("generate() ready.")

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

Loaded: Gemma 4 E2B-it (local Kaggle input, 4-bit)
chat_template present: True
EOT id: 3 | eos id: 1
generate() ready.


## Demo A — grounded, cited answers for the 5 priority panic-button scenarios

These are the **exact query strings** from the app's panic grid (`lib/ui/panic_grid.dart`), in English. Each
answer is generated the same way `GemmaService.answer()` does it in the app: retrieve top-k cards, build the
grounded prompt, generate — and each answer must end with a `[Source Name]` citation and the bilingual
disclaimer, per the prompt rules above.

In [8]:
PANIC_SCENARIOS = [
    ("Tear Gas", "There is tear gas. What do I do right now?"),
    ("Being Followed", "I am being followed. How do I get to safety?"),
    ("Someone Was Grabbed", "I witnessed someone being forced into a vehicle. What should I do?"),
    ("Network Down", "The network is down. How can I reach my family and contacts?"),
    ("Prepare My Phone", "How do I prepare my phone for safety at a protest?"),
]

for label, query in PANIC_SCENARIOS:
    print("=" * 90)
    print(f"PANIC BUTTON: {label}")
    print(f"QUERY: {query}")
    print("-" * 90)
    retrieved = retrieve_top_k(query, lang="en")
    context = format_for_prompt(retrieved, "en")
    prompt = grounded_answer_prompt(query=query, retrieved_cards=context, language="en")
    answer = generate(prompt)
    print(f"[retrieved cards: {[c['id'] for c in retrieved]}]\n")
    print(answer)
    print()


PANIC BUTTON: Tear Gas
QUERY: There is tear gas. What do I do right now?
------------------------------------------------------------------------------------------
[retrieved cards: ['teargas-001', 'teargas-002', 'pepperspray-001', 'eyeinjury-001']]

1. Move UPWIND and UPHILL immediately — get to open air away from the cloud.
2. Do NOT rub your eyes — rubbing spreads the agent and worsens burning.
3. Blink rapidly to produce tears; flush with large amounts of clean water for 10–15 minutes if available.
4. Remove contaminated clothing (top to bottom, avoiding the face); place in a sealed bag.
5. If breathing is severely affected, sit upright and take slow deep breaths in fresh air.
6. Seek medical help if breathing difficulty, severe eye pain, or skin blistering persists.

[1] Source [1]
_Guidance only, not a guarantee. Not legal or medical advice._

PANIC BUTTON: Being Followed
QUERY: I am being followed. How do I get to safety?
---------------------------------------------------------

## Demo B — the same query in English, Kiswahili, and Sheng

Same scenario (tear gas), same retrieved cards — only the `language` parameter changes, exactly as the app's
language toggle (`lib/ui/settings_screen.dart`) does. This exercises `language_instruction()` for all three
supported registers.

In [9]:
TRILINGUAL_QUERIES = {
    "en": "There is tear gas. What do I do right now?",
    "sw": "Kuna machozi ya kutoa. Nifanye nini?",
    "sheng": "Kuna tear gas hapa. Nisaidie!",
}

for lang, query in TRILINGUAL_QUERIES.items():
    print("=" * 90)
    print(f"LANGUAGE: {lang}    QUERY: {query}")
    print("-" * 90)
    retrieved = retrieve_top_k(query, lang=lang)
    context = format_for_prompt(retrieved, lang)
    prompt = grounded_answer_prompt(query=query, retrieved_cards=context, language=lang)
    answer = generate(prompt)
    print(answer)
    print()


LANGUAGE: en    QUERY: There is tear gas. What do I do right now?
------------------------------------------------------------------------------------------
1. Move UPWIND and UPHILL immediately — get to open air away from the cloud.
2. Do NOT rub your eyes — rubbing spreads the agent and worsens burning.
3. Blink rapidly to produce tears; flush with large amounts of clean water for 10–15 minutes if available.
4. Remove contaminated clothing (top to bottom, avoiding the face); place in a sealed bag.
5. If breathing is severely affected, sit upright and take slow deep breaths in fresh air.
6. Seek medical help if breathing difficulty, severe eye pain, or skin blistering persists.

[1] Source [1]
_Guidance only, not a guarantee. Not legal or medical advice._

LANGUAGE: sw    QUERY: Kuna machozi ya kutoa. Nifanye nini?
------------------------------------------------------------------------------------------
1. Songa KUELEKEA UPEPO na MAHALI PA JUU — toka nje ya wingu la gesi.
2. Blink ha

## Demo C — the refusal path

The app's rule is: if retrieval finds nothing above threshold, `GemmaService.answer()` returns the fixed refusal
string directly and **never calls the model at all** (see `gemma_service.dart`'s `answer()` — the `cards.isEmpty`
branch returns before `_streamGenerate` is reached). We reproduce that same short-circuit here: an out-of-corpus
question about something this safety corpus has no cards for should get "no verified guidance," not an improvised
answer from the model's general knowledge.

In [10]:
OUT_OF_CORPUS_QUERY = "What is the best recipe for chapati?"

retrieved = retrieve_top_k(OUT_OF_CORPUS_QUERY, lang="en")
print(f"QUERY: {OUT_OF_CORPUS_QUERY}")
print(f"Retrieved cards: {retrieved}  (empty -> refusal, model is never called)\n")

if not retrieved:
    refusal = ("I don't have that in my library. Please consult a doctor, lawyer, or trusted authority.\n\n"
               "_Guidance only, not a guarantee. Not legal or medical advice._")
    print(refusal)
else:
    print("(unexpected: retrieval found cards for an out-of-corpus query)")


QUERY: What is the best recipe for chapati?
Retrieved cards: []  (empty -> refusal, model is never called)

I don't have that in my library. Please consult a doctor, lawyer, or trusted authority.

_Guidance only, not a guarantee. Not legal or medical advice._


## Demo D — incident extraction: messy free text → strict JSON

Mirrors `ExtractorService.extract()` in `extractor.dart`: run the extraction prompt, strip any markdown fences the
model might add, parse as JSON (falling back to regex-extracting the first `{...}` block if the model wrapped the
JSON in prose), and validate the result has exactly the schema fields the app's `IncidentReport.fromJson` expects.

In [11]:
MESSY_ACCOUNT = (
    "so like around 7pm today near the Uhuru Park entrance on Kenyatta Ave, i saw these guys in "
    "plain clothes just grab my neighbour Brian off the street and shove him into a white probox, "
    "no plate number i could see, there were like 3 of them, one was wearing a red cap, my friend "
    "Wanjiru was there too and saw it happen"
)

prompt = extract_incident_prompt(MESSY_ACCOUNT)
raw = generate(prompt, max_new_tokens=300)

print("RAW MODEL OUTPUT:")
print(raw)
print()

# Same cleanup as extractor.dart: strip markdown fences, then parse (with a
# regex fallback that extracts the first {...} block if the model added prose).
import re as _re

cleaned = raw.replace("```json", "").replace("```", "").strip()
try:
    parsed = json.loads(cleaned)
except json.JSONDecodeError:
    match = _re.search(r"\{[\s\S]*\}", cleaned)
    if not match:
        raise ValueError(f"Model did not return valid JSON: {cleaned}")
    parsed = json.loads(match.group(0))

REQUIRED_FIELDS = ["who", "what", "where", "when", "details", "witnesses", "source_reliability"]
missing = [f for f in REQUIRED_FIELDS if f not in parsed]

print("PARSED (schema-validated against IncidentReport.fromJson fields):")
print(json.dumps(parsed, indent=2, ensure_ascii=False))
print(f"\nMissing required fields: {missing if missing else 'none'}")


RAW MODEL OUTPUT:
```json
{
  "who": "my neighbour Brian",
  "what": "were grabbed off the street and shoved into a white box",
  "where": "Uhuru Park entrance on Kenyatta Ave",
  "when": "around 7pm today",
  "details": "the action involved three individuals in plain clothes; one of them was wearing a red cap.",
  "witnesses": "Wanjiru",
  "source_reliability": "firsthand"
}
```

PARSED (schema-validated against IncidentReport.fromJson fields):
{
  "who": "my neighbour Brian",
  "what": "were grabbed off the street and shoved into a white box",
  "where": "Uhuru Park entrance on Kenyatta Ave",
  "when": "around 7pm today",
  "details": "the action involved three individuals in plain clothes; one of them was wearing a red cap.",
  "witnesses": "Wanjiru",
  "source_reliability": "firsthand"
}

Missing required fields: none


## Summary

This notebook ran, top-to-bottom, on a single Kaggle GPU session:

- The real 41-card verified bilingual corpus (`assets/corpus/cards.jsonl`), byte-identical to what ships in the app
- Retrieval logic ported line-for-line from `lib/services/rag.dart`
- Prompt templates ported line-for-line from `lib/prompts/prompts.dart`
- Grounded, cited, disclaimer-terminated answers for all 5 priority panic-button scenarios
- The same scenario answered in English, Kiswahili, and Sheng
- The refusal path proven to short-circuit before the model is even called, for an out-of-corpus question
- Free-text incident extraction into the app's exact structured JSON schema

**What this notebook is not:** proof of the on-device offline guarantee. The app is architected for it (`flutter_gemma/flutter_gemma_litertlm`, .litertlm weights, zero network in the request path) and the engine layer is complete, but device verification is still pending hardware — see Limitations in the writeup. What this notebook does prove is narrower and checkable: the app's actual retrieval and prompt logic, against the actual corpus, produces grounded, cited, trilingual, honestly-refusing guidance.
